# Profitability

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = "profitability"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_FIG_DIR / "fixed_profit").mkdir(parents=True, exist_ok=True)
    (OUT_FIG_DIR / "fixed_horizon").mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_TAB_DIR / "fixed_profit").mkdir(parents=True, exist_ok=True)
    (OUT_TAB_DIR / "fixed_horizon").mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_EXP_DIR / "fixed_profit").mkdir(parents=True, exist_ok=True)
    (OUT_EXP_DIR / "fixed_horizon").mkdir(parents=True, exist_ok=True)

# Common

In [ ]:
import datetime
from typing import get_args

import pandas as pd

from thesis_project.profitability import (
    fixed_horizon,
    fixed_profit,
    settings,
)

# Main

## Fixed horizon

In [ ]:
horizons = [
    10,
    30,
    1 * 60,
    5 * 60,
    10 * 60,
    15 * 60,
    30 * 60,
    45 * 60,
    1 * 60**2,
    2 * 60**2,
    3 * 60**2,
    24 * 60**2,
    3 * 24 * 60**2,
    5 * 24 * 60**2,
]

summaries = {}
for price_mode in get_args(settings.PriceMode):
    for contract_mode in get_args(settings.FutContractMode):
        for volume_mode in ("ignore", "level"):
            data = []
            for horizon in horizons:
                summary = fixed_horizon.load_experiment(
                    "fbtp",
                    price_mode,
                    contract_mode,
                    volume_mode,
                    horizon,
                )
                data.append(summary)
            summaries[(price_mode, contract_mode, volume_mode)] = pd.concat(data)

In [ ]:
horizons_to_drop = [
    30,
    45 * 60,
    2 * 60**2,
    3 * 24 * 60**2,
    5 * 24 * 60**2,
]

columns_to_drop = [
    "n_days",
    "trading_days",
    "total_pnl",
    "mean_pnl",
    "median_pnl",
    "std_pnl",
    "max_pnl",
    "min_pnl",
    "profitable_min_pnl",
    "episode_min_pnl",
    "min_episode_length",
]

for k, v in summaries.items():
    assert isinstance(v, pd.DataFrame)
    v = v.drop(index=horizons_to_drop, level="horizon")
    v = v.drop(columns=columns_to_drop)
    v = v.rename(columns={c: c.replace("_", " ") for c in v.columns})
    v = v.rename(columns={c: c.replace("pct", "(%)") for c in v.columns})
    v = v.rename(index=lambda x: pd.Timedelta(x, unit="s"), level="horizon")
    time_columns = v.columns[v.columns.str.contains("length", case=False, na=False)]
    v[time_columns] = v[time_columns].map(lambda x: str(datetime.timedelta(seconds=round(x))))
    print(k)
    display(v.round(2))

## Fixed profit

In [ ]:
profits = [
    100.0,
    500.0,
    1_000.0,
    2_500.0,
    5_000.0,
    10_000.0,
    25_000.0,
    50_000.0,
]

summaries = {}
for price_mode in get_args(settings.PriceMode):
    for contract_mode in get_args(settings.FutContractMode):
        for volume_mode in ("ignore", "level"):
            data = []
            for profit in profits:
                summary = fixed_profit.load_experiment(
                    "fbtp",
                    price_mode,
                    contract_mode,
                    volume_mode,
                    profit,
                )
                data.append(summary)
            summaries[(price_mode, contract_mode, volume_mode)] = pd.concat(data)

In [ ]:
profits_to_drop = [
    5_000.0,
    10_000.0,
    25_000.0,
    50_000.0,
]

columns_to_drop = [
    "candidate_entries",
    "max_mae",
    "max_time_to_profit",
]

for k, v in summaries.items():
    v = v.drop(index=profits_to_drop, level="profit")
    v = v.drop(columns=columns_to_drop)
    v = v.rename(columns={c: c.replace("_", " ") for c in v.columns})
    v = v.rename(columns={c: c.replace("pct", "(%)") for c in v.columns})
    v = v.rename(columns={c: c.replace("mae", "worst loss") for c in v.columns})
    time_columns = v.columns[v.columns.str.contains("time", case=False, na=False)]
    v[time_columns] = v[time_columns].map(lambda x: str(datetime.timedelta(seconds=round(x))))

    print(k)
    display(v.round(2))